In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio


pio.templates.default = "plotly_white"

print("Plotly version:", __import__("plotly").__version__)

Plotly version: 7.0.0


## 1. Load and inspect the data

In [3]:
df = pd.read_csv("earthquake_1995-2023.csv")
print("shape:", df.shape)
df.head()

shape: (1000, 19)


,title,magnitude,date_time,cdi,mmi,alert,tsunami,sig,net,nst,dmin,gap,magType,depth,latitude,longitude,location,continent,country
0,"M 6.5 - 42 km W of Sola, Vanuatu",6.5,16-08-2023 12:47,7,4,green,0,657,us,114,7.177000,25.0,mww,192.955,-13.8814,167.1580,"Sola, Vanuatu",NaN,Vanuatu
1,"M 6.5 - 43 km S of Intipucá, El Salvador",6.5,19-07-2023 00:22,8,6,yellow,0,775,us,92,0.679000,40.0,mww,69.727,12.8140,-88.1265,"Intipucá, El Salvador",NaN,NaN
2,"M 6.6 - 25 km ESE of Loncopué, Argentina",6.6,17-07-2023 03:05,7,5,green,0,899,us,70,1.634000,28.0,mww,171.371,-38.1911,-70.3731,"Loncopué, Argentina",South America,Argentina
3,"M 7.2 - 98 km S of Sand Point, Alaska",7.2,16-07-2023 06:48,6,6,green,1,860,us,173,0.907000,36.0,mww,32.571,54.3844,-160.6990,"Sand Point, Alaska",NaN,NaN
4,M 7.3 - Alaska Peninsula,7.3,16-07-2023 06:48,0,5,NaN,1,820,at,79,0.879451,172.8,Mi,21.000,54.4900,-160.7960,Alaska Peninsula,NaN,NaN


In [4]:
print("Exact duplicate rows:", df.duplicated().sum())

Exact duplicate rows: 0


In [5]:
key_dupes = df.duplicated(subset=["title", "date_time"]).sum()
print("Duplicate earthquakes (by title + date_time):", key_dupes)

Duplicate earthquakes (by title + date_time): 2


In [6]:
print("Rows before removing duplicates:", df.shape[0])

df = df.drop_duplicates(subset=["title", "date_time"])

print("Rows after removing duplicates:", df.shape[0])

Rows before removing duplicates: 1000
Rows after removing duplicates: 998


In [7]:
missing_count = df.isnull().sum()
missing_percent = (df.isnull().sum() / len(df)) * 100

missing_report = pd.DataFrame({
    "Missing Count": missing_count,
    "Missing %": missing_percent.round(2)
})

missing_report = missing_report[missing_report["Missing Count"] > 0].sort_values("Missing Count", ascending=False)
print(missing_report)

           Missing Count  Missing %
continent            714      71.54
alert                549      55.01
country              347      34.77
location               6       0.60


In [8]:
for col in ["cdi", "gap", "nst", "dmin"]:
    zero_count = (df[col] == 0).sum()
    print(f"{col}: {zero_count} zeros out of {len(df)}")

cdi: 395 zeros out of 998
gap: 252 zeros out of 998
nst: 519 zeros out of 998
dmin: 586 zeros out of 998


## 1.1 Univariate histogram of magnitude with a range slider

In [ ]:
f1 = px.histogram(df, x="magnitude",
                  title="Distribution of Earthquake (1995 to 2023)",
                  labels={"magnitude": "Magnitude"})

f1.update_traces(xbins=dict(start=6.45, end=9.15, size=0.1))
f1.update_layout(yaxis_title="Number of events", bargap=0.05,
                 xaxis=dict(rangeslider=dict(visible=True, thickness=0.12), dtick=0.5))

f1.show()

## 1.2 Multivariate Scatter: magnitude vs depth, coloured by alert level

In [14]:
# Define alert levels
ALERT_ORDER = ["green", "yellow", "orange", "red"]

# Define colors for each alert level
ALERT_COLORS = {
    "green": "green",
    "yellow": "yellow",
    "orange": "orange",
    "red": "red"
}

# Define depth tick values
DEPTH_TICKS = [1, 10, 100, 1000]

print(df["alert"].unique())

['green' 'yellow' nan 'orange' 'red']


In [15]:
fig2 = px.scatter(
    df,
    x="magnitude",
    y="depth",
    color="alert",
    category_orders={"alert": ALERT_ORDER},
    color_discrete_map=ALERT_COLORS,
    hover_name="title",
    hover_data={
        "location": True,
        "country": True,
        "date_time": "|%d %b %Y %H:%M",
        "sig": True,
        "alert": False
    },
    log_y=True,
    opacity=0.75,
    labels={
        "magnitude": "Magnitude",
        "depth": "Depth (km, log scale)",
        "alert": "Alert level"
    },
    title="Magnitude vs. depth, coloured by alert level"
)

fig2.update_yaxes(
    autorange="reversed",
    tickvals=DEPTH_TICKS
)

fig2.show()